In [16]:
uv pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


Using Python 3.13.7 environment at: c:\Users\ddddd\AppData\Local\Programs\Python\Python313
Audited 1 package in 3ms


In [ ]:
import re
import polars as pl
import plotly.express as px

df = pl.read_parquet("spy_10k_2015_present.parquet")

# Baseline ESG keywords/phrases
keywords = ["environment", "sustainability", "climate change", "carbon emissions"]

# Build regex patterns:
# - case-insensitive
# - word boundaries for single words
# - allow whitespace/hyphen between words in phrases (e.g., "climate-change")
patterns = {}
for k in keywords:
    if " " in k:
        parts = [re.escape(p) for p in k.split()]
        pat = r"(?i)\b" + r"[-\s]+".join(parts) + r"\b"
    else:
        pat = r"(?i)\b" + re.escape(k) + r"\b"
    patterns[k] = pat

wordcount_df = df.with_columns(
    [pl.col("text").str.count_matches(patterns[k]).alias(f"{k}_count") for k in keywords]
)

ticker = "AAPL"
section = "risk_factors"

plot_df = (
    wordcount_df
    .filter((pl.col("ticker") == ticker) & (pl.col("section") == section))
    .sort("filing_date")
    .select(["ticker", "filing_date"] + [f"{k}_count" for k in keywords])
    .to_pandas()
)

fig = px.line(
    plot_df,
    x="filing_date",
    y=[f"{k}_count" for k in keywords],
    title=f"Keyword Occurrences in 10-K Filings for {ticker} ({section})",
    labels={"filing_date": "Filing Date", "value": "Count"},
    markers=True,
)
fig.update_layout(legend_title_text="Keywords", template="plotly_white")
fig.update_yaxes(
    tickmode="linear",
    tick0=0,
    dtick=1
)

fig.show()
